# 🧹 PHẦN 2: TIỀN XỬ LÝ & ÉP KIỂU DỮ LIỆU (DATA CLEANING & TYPE CASTING)
## File: `02_Data_Cleaning_And_Type_Casting.ipynb`
---
**Mục tiêu của phần này:**
1. Đọc lại file dữ liệu thô `data/raw/raw_movies_2000_2026.csv` vừa cào ở Phần 1.
2. Kiểm tra chi tiết cấu trúc bảng, kiểu dữ liệu hiện tại (`dtypes`), các giá trị khuyết thiếu và ký tự `\N`.
3. **Ép kiểu dữ liệu (Type Casting)**:
   - `year`: chuỗi $\rightarrow$ số nguyên `int32`.
   - `runtime_minutes`: thay `\N` bằng NaN, điền trung vị (`median`), ép kiểu `int32`.
   - `rating`: ép kiểu `float64`.
   - `votes`: ép kiểu `int64`.
4. **Làm sạch thể loại (`genres`)**: thay `\N` bằng `Unknown`, tách list thể loại `genres_list`.
5. Bổ sung các trường liên kết phát phim `watch_url` và `imdb_url`.
6. Lưu dữ liệu đã làm sạch vào `data/interim/cleaned_movies_2000_2026.csv`.

### 1. Khai báo thư viện & Đọc dữ liệu thô

In [ ]:
import os
import pandas as pd
import numpy as np

# Cấu hình hiển thị bảng Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

# Đọc dữ liệu thô từ file cào ở Phần 1
raw_path = 'data/raw/raw_movies_2000_2026.csv'
df = pd.read_csv(raw_path)
print(f'✅ Đã nạp thành công {len(df):,} dòng dữ liệu thô từ {raw_path}!')

### 2. Kiểm tra cấu trúc & Phát hiện vấn đề về kiểu dữ liệu (Data Inspection)

In [ ]:
print('--- KIỂU DỮ LIỆU BAN ĐẦU CỦA CÁC CỘT (DTYPES) ---')
print(df.dtypes)

print('\n--- TỔNG QUAN BẢNG DỮ LIỆU THÔ ---')
df.info()

print('\n--- ĐẾM CÁC DÒNG CÓ KÝ TỰ THIẾU \\N TRONG DỮ LIỆU IMDB ---')
for col in df.columns:
    count_n = (df[col].astype(str) == r'\N').sum()
    if count_n > 0:
        print(f'Cột "{col}" có {count_n:,} dòng chứa ký tự "\\N" cần xử lý!')

### 3. Thực hiện Ép kiểu dữ liệu (Type Casting) & Xử lý Missing Values

In [ ]:
# 1. Ép kiểu năm phát hành (year) sang số nguyên int32
df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('int32')

# 2. Xử lý thời lượng phim (runtime_minutes):
# - Thay '\N' bằng NaN
# - Chuyển sang số
# - Điền giá trị trung vị (median) cho phim thiếu thời lượng
df['runtime_minutes'] = df['runtime_minutes'].replace(r'\N', np.nan)
df['runtime_minutes'] = pd.to_numeric(df['runtime_minutes'], errors='coerce')
median_runtime = df['runtime_minutes'].median()
df['runtime_minutes'] = df['runtime_minutes'].fillna(median_runtime).astype('int32')

# 3. Ép kiểu điểm số (rating) sang float64 và lượt vote sang int64
df['rating'] = pd.to_numeric(df['rating'], errors='coerce').astype('float64')
df['votes'] = pd.to_numeric(df['votes'], errors='coerce').astype('int64')

# 4. Làm sạch thể loại (genres):
# - Thay '\N' bằng 'Unknown'
# - Tạo chuỗi genres_clean (thay dấu phẩy bằng khoảng trắng)
# - Tạo cột genres_list dạng mảng Python
df['genres'] = df['genres'].replace(r'\N', 'Unknown').fillna('Unknown')
df['genres_clean'] = df['genres'].str.replace(',', ' ')
df['genres_list'] = df['genres'].apply(lambda g: [x.strip() for x in g.split(',') if x.strip()])

# 5. Bổ sung các đường dẫn hỗ trợ Web App
df['imdb_url'] = 'https://www.imdb.com/title/' + df['id'] + '/'
df['watch_url'] = 'https://2embed.cc/embed/' + df['id']

print('✅ Đã hoàn tất toàn bộ các bước ép kiểu và làm sạch!')

### 4. Kiểm tra lại dữ liệu sau khi ép kiểu & so sánh

In [ ]:
print('--- KIỂU DỮ LIỆU CHUẨN SAU KHI ÉP KIỂU (DTYPES MỚI) ---')
print(df.dtypes)

print('\n--- KIỂM TRA LẠI THÔNG TIN VÀ CÁC GIÁ TRỊ NULL ---')
df.info()

print('\n--- 5 BẢN GHI ĐẦU TIÊN SAU KHI LÀM SẠCH ---')
display(df.head(5))

### 5. Lưu dữ liệu đã làm sạch vào thư mục `data/interim/`

In [ ]:
output_dir = 'data/interim'
os.makedirs(output_dir, exist_ok=True)

clean_csv_path = os.path.join(output_dir, 'cleaned_movies_2000_2026.csv')
clean_json_path = os.path.join(output_dir, 'cleaned_movies_2000_2026.json')

df.to_csv(clean_csv_path, index=False, encoding='utf-8-sig')
df.to_json(clean_json_path, orient='records', force_ascii=False, indent=2)

print(f'✅ Đã lưu file sạch CSV: {clean_csv_path}')
print(f'✅ Đã lưu file sạch JSON: {clean_json_path}')
print('\n👉 BƯỚC TIẾP THEO: Hãy mở file `03_Exploratory_Data_Analysis.ipynb` để phân tích thống kê và vẽ biểu đồ!')